# Extração S3 (Supabase Storage): DataLakeVertice
Bucket S3 → Volume `bronze/raw` → tabelas `bronze.*`

In [1]:
import io, os

import boto3
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from dotenv import load_dotenv

load_dotenv()

S3_ENDPOINT = "https://jedxlexetmsyukoyztgr.storage.supabase.co/storage/v1/s3"
S3_REGION = "us-east-1"
S3_BUCKET = "DataLakeVertice"
RAW = "/Volumes/lakehouse_vertice/bronze/raw"

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    region_name=S3_REGION,
    aws_access_key_id=os.getenv("ACCESS_KEY"),
    aws_secret_access_key=os.getenv("SECRET_KEY"),
)
spark = DatabricksSession.builder.serverless().getOrCreate()
w = WorkspaceClient()


def arquivos_no_volume(pasta):
    try:
        entradas = list(w.files.list_directory_contents(pasta))
    except NotFound:
        return set()

    caminhos = set()
    for e in entradas:
        if e.is_directory:
            caminhos |= arquivos_no_volume(e.path.rstrip("/"))
        else:
            caminhos.add(e.path)
    return caminhos

## 1. Pastas do bucket

In [2]:
resposta = s3.list_objects_v2(Bucket=S3_BUCKET, Delimiter="/")
[p["Prefix"] for p in resposta.get("CommonPrefixes", [])]

['clickup/',
 'compras/',
 'documentos/',
 'excel/',
 'financeiro/',
 'msproject/',
 'pmo/',
 'rh/',
 'timesheet/']

## 2. Cópia incremental do bucket para o Volume (raw)

In [3]:
# só baixa o que ainda não está no Volume; arquivos já existentes nunca são sobrescritos
existentes = arquivos_no_volume(RAW)

# o S3 devolve no máximo 1000 chaves por chamada; o paginator busca as próximas páginas
paginas = s3.get_paginator("list_objects_v2").paginate(Bucket=S3_BUCKET)

enviados, pulados = 0, 0
for pagina in paginas:
    for obj in pagina.get("Contents", []):
        chave = obj["Key"]
        destino = f"{RAW}/{chave}"
        if chave.endswith("/"):
            continue
        if destino in existentes:
            pulados += 1
            continue

        conteudo = s3.get_object(Bucket=S3_BUCKET, Key=chave)["Body"].read()
        w.files.upload(destino, io.BytesIO(conteudo), overwrite=False)
        enviados += 1

print(f"{enviados} novos, {pulados} já existiam em {RAW}")

73 novos, 0 já existiam em /Volumes/lakehouse_vertice/bronze/raw


## 3. Tabelas bronze: CSV

In [4]:
csvs = {
    "projetos":            "pmo/projetos.csv",
    "status_semanal":      "pmo/status_semanal.csv",
    "colaboradores":       "rh/colaboradores.csv",
    "apontamentos":        "timesheet/apontamentos.csv",
    "custos":              "financeiro/custos.csv",
    "entregas_fornecedor": "compras/entregas_fornecedor.csv",
}

for tabela, arquivo in csvs.items():
    spark.sql(f"""
    CREATE OR REPLACE TABLE lakehouse_vertice.bronze.{tabela} AS
    SELECT * EXCEPT (_rescued_data)
    FROM read_files('{RAW}/{arquivo}',
                    format => 'csv', header => true, inferColumnTypes => false)
    """)
    print("criada:", tabela)

criada: projetos
criada: status_semanal
criada: colaboradores
criada: apontamentos
criada: custos
criada: entregas_fornecedor


## 4. Tabelas bronze: Excel, ClickUp e MS Project

In [5]:
spark.sql(f"""
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.excel_tarefas AS
SELECT `ID`               AS id,
       `Fase`             AS fase,
       `Tarefa`           AS tarefa,
       `Responsável`      AS responsavel,
       `Início previsto`  AS inicio_previsto,
       `Término previsto` AS termino_previsto,
       `Início real`      AS inicio_real,
       `Término real`     AS termino_real,
       `% Concluído`      AS pct_concluido,
       `Horas previstas`  AS horas_previstas,
       url_decode(_metadata.file_name) AS arquivo
FROM read_files('{RAW}/excel/',
                format => 'excel', headerRows => 1, inferColumnTypes => false)
""")
print("criada: excel_tarefas")

# cada JSON do ClickUp traz uma lista com um array de tarefas; explode gera uma linha por tarefa
spark.sql(f"""
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.clickup_tarefas AS
SELECT list.id   AS list_id,
       list.name AS list_name,
       t.*,
       url_decode(_metadata.file_name) AS arquivo
FROM (
    SELECT list, explode(tasks) AS t, _metadata
    FROM read_files('{RAW}/clickup/',
                    format => 'json', multiLine => true)
)
""")
print("criada: clickup_tarefas")

spark.sql(f"""
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.msproject_tarefas AS
SELECT * EXCEPT (_rescued_data),
       url_decode(_metadata.file_name) AS arquivo
FROM read_files('{RAW}/msproject/',
                format => 'xml', rowTag => 'Task')
""")
print("criada: msproject_tarefas")

criada: excel_tarefas
criada: clickup_tarefas
criada: msproject_tarefas


## 5. Conferência
Esperado: apontamentos 30.883 · custos 23.640 · status_semanal 3.911 · clickup_tarefas 2.119 · excel_tarefas 1.663 · msproject_tarefas 1.260 · entregas_fornecedor 640 · colaboradores 523 · projetos 60

In [6]:
spark.sql("""
SELECT 'projetos' AS tabela, COUNT(*) AS linhas FROM lakehouse_vertice.bronze.projetos
UNION ALL SELECT 'status_semanal', COUNT(*) FROM lakehouse_vertice.bronze.status_semanal
UNION ALL SELECT 'excel_tarefas', COUNT(*) FROM lakehouse_vertice.bronze.excel_tarefas
UNION ALL SELECT 'clickup_tarefas', COUNT(*) FROM lakehouse_vertice.bronze.clickup_tarefas
UNION ALL SELECT 'msproject_tarefas', COUNT(*) FROM lakehouse_vertice.bronze.msproject_tarefas
UNION ALL SELECT 'colaboradores', COUNT(*) FROM lakehouse_vertice.bronze.colaboradores
UNION ALL SELECT 'apontamentos', COUNT(*) FROM lakehouse_vertice.bronze.apontamentos
UNION ALL SELECT 'custos', COUNT(*) FROM lakehouse_vertice.bronze.custos
UNION ALL SELECT 'entregas_fornecedor', COUNT(*) FROM lakehouse_vertice.bronze.entregas_fornecedor
ORDER BY linhas DESC
""").show()

+-------------------+------+
|             tabela|linhas|
+-------------------+------+
|       apontamentos| 30883|
|             custos| 23640|
|     status_semanal|  3911|
|    clickup_tarefas|  2119|
|      excel_tarefas|  1500|
|  msproject_tarefas|  1260|
|entregas_fornecedor|   640|
|      colaboradores|   523|
|           projetos|    60|
+-------------------+------+

